In [1]:
import nltk
import pandas as pd
import emoji
import pyarabic.araby as araby
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer
from nltk.tokenize import word_tokenize

import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

import qalsadi.lemmatizer

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\osamo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\osamo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\osamo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
df = pd.read_csv('AAFAQ_Dataset.csv')
df

,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.
...,...,...,...
5004,ماذا يحدث عند تسخين الحديد؟,العلوم,عند تسخين الحديد، تتسارع حركة جزيئاته مما يؤدي...
5005,ما هو الهدف من اتفاقية التجارة الحرة لأمريكا ا...,الاقتصاد والعمل,الهدف من اتفاقية التجارة الحرة لأمريكا الشمالي...
5006,ما هو الغرض من اتفاقية التجارة الحرة لأمريكا ا...,الاقتصاد والعمل,الغرض من اتفاقية التجارة الحرة لأمريكا الشمالي...
5007,ما هو هدف إعجاز القرآن وفقًا للمعتقد الإسلامي؟,الدين,يؤدي الإعجاز غرضين رئيسين الأول وهو أثبات أصال...


In [3]:
testing_df = df.copy()
testing_df["Original_QuestionText"] = testing_df["QuestionText"]
testing_df["Original_Answer"] = testing_df["Answer"]
testing_df = testing_df.drop(columns=["Category"])

Preparing techniques' functions

In [5]:
def remove_emoji(text):
    text = str(text)
    text = emoji.replace_emoji(text, replace='')
    return text


def remove_tashkeel(text):
    text = str(text)
    text = araby.strip_tashkeel(text)
    return text


def remove_tatweel(text):
    text = str(text)
    text = araby.strip_tatweel(text)
    return text


stop_words = set(stopwords.words('arabic'))
def remove_stopwords(text):
    text = str(text)
    words = text.split()
    filtered_words = []

    for word in words:
        if word not in stop_words:
            filtered_words.append(word)
    cleaned_text = ' '.join(filtered_words)

    return cleaned_text


def normalize_hamza(text):
    text = str(text)
    text = araby.normalize_hamza(text)
    return text


def tokenize_text(text):
    text = str(text)
    tokens = word_tokenize(text)
    return tokens


def remove_punctation(tokens):
    clean_tokens = []

    for token in tokens:
        if token.isalnum():
            clean_tokens.append(token)

    return clean_tokens


stemmer = ISRIStemmer()

def stem_tokens(tokens):
    stemmed_tokens = []

    for token in tokens:
        stemmed_tokens.append(stemmer.stem(token))

    return stemmed_tokens


lemmer = qalsadi.lemmatizer.Lemmatizer()
def arabic_lemmatization(text):
    text = str(text)
    words = text.split()
    lemmas = []

    for word in words:
        lemma = lemmer.lemmatize(word)
        lemmas.append(lemma)

    return " ".join(lemmas)

Checking for emojis

In [6]:
mask_emoji_question = testing_df['QuestionText'].astype(str).apply(
    lambda text: any(char in emoji.EMOJI_DATA for char in text)
)

mask_emoji_answer = testing_df['Answer'].astype(str).apply(
    lambda text: any(char in emoji.EMOJI_DATA for char in text)
)

print("Emojis in QuestionText:", mask_emoji_question.sum())
print("Emojis in Answer:", mask_emoji_answer.sum())

Emojis in QuestionText: 0
Emojis in Answer: 0


Removing tashkeel

In [8]:
testing_df = df.copy()
testing_df["Original_QuestionText"] = testing_df["QuestionText"]
testing_df["Original_Answer"] = testing_df["Answer"]
testing_df = testing_df.drop(columns=["Category"])

In [9]:
testing_df['QuestionText'] = testing_df['QuestionText'].apply(remove_tashkeel)
testing_df['Answer'] = testing_df['Answer'].apply(remove_tashkeel)
testing_df

,QuestionText,Answer,Original_QuestionText,Original_Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.,أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تعرف بأنها كائنات حية دقيقة.,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساسا من النيتروجين؟,الهواء يتكون أساسا من النيتروجين.,أيتكون الهواء أساساً من النيتروجين؟,الهواء يتكون أساساً من النيتروجين.
...,...,...,...,...
5004,ماذا يحدث عند تسخين الحديد؟,عند تسخين الحديد، تتسارع حركة جزيئاته مما يؤدي...,ماذا يحدث عند تسخين الحديد؟,عند تسخين الحديد، تتسارع حركة جزيئاته مما يؤدي...
5005,ما هو الهدف من اتفاقية التجارة الحرة لأمريكا ا...,الهدف من اتفاقية التجارة الحرة لأمريكا الشمالي...,ما هو الهدف من اتفاقية التجارة الحرة لأمريكا ا...,الهدف من اتفاقية التجارة الحرة لأمريكا الشمالي...
5006,ما هو الغرض من اتفاقية التجارة الحرة لأمريكا ا...,الغرض من اتفاقية التجارة الحرة لأمريكا الشمالي...,ما هو الغرض من اتفاقية التجارة الحرة لأمريكا ا...,الغرض من اتفاقية التجارة الحرة لأمريكا الشمالي...
5007,ما هو هدف إعجاز القرآن وفقا للمعتقد الإسلامي؟,يؤدي الإعجاز غرضين رئيسين الأول وهو أثبات أصال...,ما هو هدف إعجاز القرآن وفقًا للمعتقد الإسلامي؟,يؤدي الإعجاز غرضين رئيسين الأول وهو أثبات أصال...


Tatweel Removal

In [11]:
testing_df = df.copy()
testing_df["Original_QuestionText"] = testing_df["QuestionText"]
testing_df["Original_Answer"] = testing_df["Answer"]
testing_df = testing_df.drop(columns=["Category"])

In [12]:
mask_tatweel_question = testing_df['QuestionText'].astype(str).str.contains('ـ', regex=False)
mask_tatweel_answer = testing_df['Answer'].astype(str).str.contains('ـ', regex=False)

print("Tatweel in QuestionText:", mask_tatweel_question.sum())
print("Tatweel in Answer:", mask_tatweel_answer.sum())

Tatweel in QuestionText: 6
Tatweel in Answer: 33


In [13]:
testing_df['QuestionText'] = testing_df['QuestionText'].apply(remove_tatweel)
testing_df['Answer'] = testing_df['Answer'].apply(remove_tatweel)
testing_df

,QuestionText,Answer,Original_QuestionText,Original_Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.,أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تُعرف بأنها كائنات حية دقيقة.,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,الهواء يتكون أساساً من النيتروجين.,أيتكون الهواء أساساً من النيتروجين؟,الهواء يتكون أساساً من النيتروجين.
...,...,...,...,...
5004,ماذا يحدث عند تسخين الحديد؟,عند تسخين الحديد، تتسارع حركة جزيئاته مما يؤدي...,ماذا يحدث عند تسخين الحديد؟,عند تسخين الحديد، تتسارع حركة جزيئاته مما يؤدي...
5005,ما هو الهدف من اتفاقية التجارة الحرة لأمريكا ا...,الهدف من اتفاقية التجارة الحرة لأمريكا الشمالي...,ما هو الهدف من اتفاقية التجارة الحرة لأمريكا ا...,الهدف من اتفاقية التجارة الحرة لأمريكا الشمالي...
5006,ما هو الغرض من اتفاقية التجارة الحرة لأمريكا ا...,الغرض من اتفاقية التجارة الحرة لأمريكا الشمالي...,ما هو الغرض من اتفاقية التجارة الحرة لأمريكا ا...,الغرض من اتفاقية التجارة الحرة لأمريكا الشمالي...
5007,ما هو هدف إعجاز القرآن وفقًا للمعتقد الإسلامي؟,يؤدي الإعجاز غرضين رئيسين الأول وهو أثبات أصال...,ما هو هدف إعجاز القرآن وفقًا للمعتقد الإسلامي؟,يؤدي الإعجاز غرضين رئيسين الأول وهو أثبات أصال...


Normalize hamza

In [14]:
testing_df = df.copy()
testing_df["Original_QuestionText"] = testing_df["QuestionText"]
testing_df["Original_Answer"] = testing_df["Answer"]
testing_df = testing_df.drop(columns=["Category"])

In [15]:
testing_df['QuestionText'] = testing_df['QuestionText'].apply(normalize_hamza)
testing_df['Answer'] = testing_df['Answer'].apply(normalize_hamza)

testing_df

,QuestionText,Answer,Original_QuestionText,Original_Answer
0,ءيهما ءفضل الدراسة في السابق ءم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر ءفضل بسبب توفر ا...,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,ءليس القطن عماد الثروة في مصر؟,القطن يعتبر من ءهم المنتجات الزراعية في مصر، و...,أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,ءتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.,أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.
3,ءتعرف البكتيريا بءنها كاءنات حية دقيقة؟,البكتيريا تُعرف بءنها كاءنات حية دقيقة.,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,ءيتكون الهواء ءساساً من النيتروجين؟,الهواء يتكون ءساساً من النيتروجين.,أيتكون الهواء أساساً من النيتروجين؟,الهواء يتكون أساساً من النيتروجين.
...,...,...,...,...
5004,ماذا يحدث عند تسخين الحديد؟,عند تسخين الحديد، تتسارع حركة جزيءاته مما يءدي...,ماذا يحدث عند تسخين الحديد؟,عند تسخين الحديد، تتسارع حركة جزيئاته مما يؤدي...
5005,ما هو الهدف من اتفاقية التجارة الحرة لءمريكا ا...,الهدف من اتفاقية التجارة الحرة لءمريكا الشمالي...,ما هو الهدف من اتفاقية التجارة الحرة لأمريكا ا...,الهدف من اتفاقية التجارة الحرة لأمريكا الشمالي...
5006,ما هو الغرض من اتفاقية التجارة الحرة لءمريكا ا...,الغرض من اتفاقية التجارة الحرة لءمريكا الشمالي...,ما هو الغرض من اتفاقية التجارة الحرة لأمريكا ا...,الغرض من اتفاقية التجارة الحرة لأمريكا الشمالي...
5007,ما هو هدف ءعجاز القرءءن وفقًا للمعتقد الءسلامي؟,يءدي الءعجاز غرضين رءيسين الءول وهو ءثبات ءصال...,ما هو هدف إعجاز القرآن وفقًا للمعتقد الإسلامي؟,يؤدي الإعجاز غرضين رئيسين الأول وهو أثبات أصال...


Removing stop words

In [16]:
testing_df = df.copy()
testing_df["Original_QuestionText"] = testing_df["QuestionText"]
testing_df["Original_Answer"] = testing_df["Answer"]
testing_df = testing_df.drop(columns=["Category"])

In [18]:
testing_df['QuestionText'] = testing_df['QuestionText'].apply(remove_stopwords)
testing_df['Answer'] = testing_df['Answer'].apply(remove_stopwords)

testing_df

,QuestionText,Answer,Original_QuestionText,Original_Answer
0,أيهما أفضل الدراسة السابق الوقت الحالي؟,الدراسة الوقت الحالي تعتبر أفضل بسبب توفر التك...,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة مصر؟,القطن يعتبر أهم المنتجات الزراعية مصر، ويعد ال...,أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس الشرق؟,الشمس تصعد الشرق.,أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تُعرف بأنها كائنات حية دقيقة.,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً النيتروجين؟,الهواء يتكون أساساً النيتروجين.,أيتكون الهواء أساساً من النيتروجين؟,الهواء يتكون أساساً من النيتروجين.
...,...,...,...,...
5004,يحدث تسخين الحديد؟,تسخين الحديد، تتسارع حركة جزيئاته يؤدي تمدده و...,ماذا يحدث عند تسخين الحديد؟,عند تسخين الحديد، تتسارع حركة جزيئاته مما يؤدي...
5005,الهدف اتفاقية التجارة الحرة لأمريكا الشمالية؟,الهدف اتفاقية التجارة الحرة لأمريكا الشمالية (...,ما هو الهدف من اتفاقية التجارة الحرة لأمريكا ا...,الهدف من اتفاقية التجارة الحرة لأمريكا الشمالي...
5006,الغرض اتفاقية التجارة الحرة لأمريكا الشمالية؟,الغرض اتفاقية التجارة الحرة لأمريكا الشمالية (...,ما هو الغرض من اتفاقية التجارة الحرة لأمريكا ا...,الغرض من اتفاقية التجارة الحرة لأمريكا الشمالي...
5007,هدف إعجاز القرآن وفقًا للمعتقد الإسلامي؟,يؤدي الإعجاز غرضين رئيسين الأول أثبات أصالة ال...,ما هو هدف إعجاز القرآن وفقًا للمعتقد الإسلامي؟,يؤدي الإعجاز غرضين رئيسين الأول وهو أثبات أصال...


Tokenization

In [19]:
testing_df = df.copy()
testing_df["Original_QuestionText"] = testing_df["QuestionText"]
testing_df["Original_Answer"] = testing_df["Answer"]
testing_df = testing_df.drop(columns=["Category"])

In [20]:
testing_df["QuestionText"] = testing_df["QuestionText"].apply(tokenize_text)
testing_df["Answer"] = testing_df["Answer"].apply(tokenize_text)
testing_df

,QuestionText,Answer,Original_QuestionText,Original_Answer
0,"[أيهما, أفضل, الدراسة, في, السابق, أم, في, الو...","[الدراسة, في, الوقت, الحالي, تعتبر, أفضل, بسبب...",أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,"[أليس, القطن, عماد, الثروة, في, مصر؟]","[القطن, يعتبر, من, أهم, المنتجات, الزراعية, في...",أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,"[أتصعد, الشمس, من, الشرق؟]","[الشمس, تصعد, من, الشرق, .]",أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.
3,"[أتعرف, البكتيريا, بأنها, كائنات, حية, دقيقة؟]","[البكتيريا, تُعرف, بأنها, كائنات, حية, دقيقة, .]",أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,"[أيتكون, الهواء, أساساً, من, النيتروجين؟]","[الهواء, يتكون, أساساً, من, النيتروجين, .]",أيتكون الهواء أساساً من النيتروجين؟,الهواء يتكون أساساً من النيتروجين.
...,...,...,...,...
5004,"[ماذا, يحدث, عند, تسخين, الحديد؟]","[عند, تسخين, الحديد،, تتسارع, حركة, جزيئاته, م...",ماذا يحدث عند تسخين الحديد؟,عند تسخين الحديد، تتسارع حركة جزيئاته مما يؤدي...
5005,"[ما, هو, الهدف, من, اتفاقية, التجارة, الحرة, ل...","[الهدف, من, اتفاقية, التجارة, الحرة, لأمريكا, ...",ما هو الهدف من اتفاقية التجارة الحرة لأمريكا ا...,الهدف من اتفاقية التجارة الحرة لأمريكا الشمالي...
5006,"[ما, هو, الغرض, من, اتفاقية, التجارة, الحرة, ل...","[الغرض, من, اتفاقية, التجارة, الحرة, لأمريكا, ...",ما هو الغرض من اتفاقية التجارة الحرة لأمريكا ا...,الغرض من اتفاقية التجارة الحرة لأمريكا الشمالي...
5007,"[ما, هو, هدف, إعجاز, القرآن, وفقًا, للمعتقد, ا...","[يؤدي, الإعجاز, غرضين, رئيسين, الأول, وهو, أثب...",ما هو هدف إعجاز القرآن وفقًا للمعتقد الإسلامي؟,يؤدي الإعجاز غرضين رئيسين الأول وهو أثبات أصال...


Remove Punctation

In [21]:
testing_df['QuestionText'] = testing_df['QuestionText'].apply(remove_punctation)
testing_df['Answer'] = testing_df['Answer'].apply(remove_punctation)

testing_df

,QuestionText,Answer,Original_QuestionText,Original_Answer
0,"[أيهما, أفضل, الدراسة, في, السابق, أم, في, الوقت]","[الدراسة, في, الوقت, الحالي, تعتبر, أفضل, بسبب...",أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,"[أليس, القطن, عماد, الثروة, في]","[القطن, يعتبر, من, أهم, المنتجات, الزراعية, في...",أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,"[أتصعد, الشمس, من]","[الشمس, تصعد, من, الشرق]",أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.
3,"[أتعرف, البكتيريا, بأنها, كائنات, حية]","[البكتيريا, بأنها, كائنات, حية, دقيقة]",أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,"[أيتكون, الهواء, من]","[الهواء, يتكون, من, النيتروجين]",أيتكون الهواء أساساً من النيتروجين؟,الهواء يتكون أساساً من النيتروجين.
...,...,...,...,...
5004,"[ماذا, يحدث, عند, تسخين]","[عند, تسخين, تتسارع, حركة, جزيئاته, مما, يؤدي,...",ماذا يحدث عند تسخين الحديد؟,عند تسخين الحديد، تتسارع حركة جزيئاته مما يؤدي...
5005,"[ما, هو, الهدف, من, اتفاقية, التجارة, الحرة, ل...","[الهدف, من, اتفاقية, التجارة, الحرة, لأمريكا, ...",ما هو الهدف من اتفاقية التجارة الحرة لأمريكا ا...,الهدف من اتفاقية التجارة الحرة لأمريكا الشمالي...
5006,"[ما, هو, الغرض, من, اتفاقية, التجارة, الحرة, ل...","[الغرض, من, اتفاقية, التجارة, الحرة, لأمريكا, ...",ما هو الغرض من اتفاقية التجارة الحرة لأمريكا ا...,الغرض من اتفاقية التجارة الحرة لأمريكا الشمالي...
5007,"[ما, هو, هدف, إعجاز, القرآن, للمعتقد]","[يؤدي, الإعجاز, غرضين, رئيسين, الأول, وهو, أثب...",ما هو هدف إعجاز القرآن وفقًا للمعتقد الإسلامي؟,يؤدي الإعجاز غرضين رئيسين الأول وهو أثبات أصال...


Stemming

In [22]:
testing_df['QuestionText'] = testing_df['QuestionText'].apply(stem_tokens)
testing_df['Answer'] = testing_df['Answer'].apply(stem_tokens)

testing_df

,QuestionText,Answer,Original_QuestionText,Original_Answer
0,"[ايه, فضل, درس, في, سبق, ام, في, وقت]","[درس, في, وقت, الحالي, عبر, فضل, سبب, وفر, كنو...",أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,"[الس, قطن, عمد, ثرة, في]","[قطن, عبر, من, اهم, نتج, زرع, في, يعد, من, عمد...",أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,"[صعد, شمس, من]","[شمس, صعد, من, شرق]",أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.
3,"[عرف, كتر, بأن, كئن, حية]","[كتر, بأن, كئن, حية, دقق]",أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,"[ايت, هوء, من]","[هوء, يتك, من, ترج]",أيتكون الهواء أساساً من النيتروجين؟,الهواء يتكون أساساً من النيتروجين.
...,...,...,...,...
5004,"[اذا, حدث, عند, تسخ]","[عند, تسخ, سرع, حرك, جزئ, مما, يؤد, الى, مدد, ...",ماذا يحدث عند تسخين الحديد؟,عند تسخين الحديد، تتسارع حركة جزيئاته مما يؤدي...
5005,"[ما, هو, هدف, من, تفق, جار, حرة, أمر]","[هدف, من, تفق, جار, حرة, أمر, شمل, NAFTA, هو, ...",ما هو الهدف من اتفاقية التجارة الحرة لأمريكا ا...,الهدف من اتفاقية التجارة الحرة لأمريكا الشمالي...
5006,"[ما, هو, غرض, من, تفق, جار, حرة, أمر]","[غرض, من, تفق, جار, حرة, أمر, شمل, NAFTA, هو, ...",ما هو الغرض من اتفاقية التجارة الحرة لأمريكا ا...,الغرض من اتفاقية التجارة الحرة لأمريكا الشمالي...
5007,"[ما, هو, هدف, عجز, قرآ, عقد]","[يؤد, عجز, غرض, رئس, اول, وهو, اثب, صلة, قرآ, ...",ما هو هدف إعجاز القرآن وفقًا للمعتقد الإسلامي؟,يؤدي الإعجاز غرضين رئيسين الأول وهو أثبات أصال...


In [23]:
# 1. Remove tashkeel
df['QuestionText'] = df['QuestionText'].apply(remove_tashkeel)
df['Answer'] = df['Answer'].apply(remove_tashkeel)

# 2. Remove tatweel
df['QuestionText'] = df['QuestionText'].apply(remove_tatweel)
df['Answer'] = df['Answer'].apply(remove_tatweel)

# 3. Remove stop words
df['QuestionText'] = df['QuestionText'].apply(remove_stopwords)
df['Answer'] = df['Answer'].apply(remove_stopwords)

# 4. Tokenization
df['QuestionText'] = df['QuestionText'].apply(tokenize_text)
df['Answer'] = df['Answer'].apply(tokenize_text)

# 5. Remove punctuation
df['QuestionText'] = df['QuestionText'].apply(remove_punctation)
df['Answer'] = df['Answer'].apply(remove_punctation)

df

,QuestionText,Category,Answer
0,"[أيهما, أفضل, الدراسة, السابق, الوقت]",التعليم,"[الدراسة, الوقت, الحالي, تعتبر, أفضل, بسبب, تو..."
1,"[أليس, القطن, عماد, الثروة]",الاقتصاد والعمل,"[القطن, يعتبر, أهم, المنتجات, الزراعية, ويعد, ..."
2,"[أتصعد, الشمس]",التعليم,"[الشمس, تصعد, الشرق]"
3,"[أتعرف, البكتيريا, بأنها, كائنات, حية]",التعليم,"[البكتيريا, تعرف, بأنها, كائنات, حية, دقيقة]"
4,"[أيتكون, الهواء, أساسا]",التعليم,"[الهواء, يتكون, أساسا, النيتروجين]"
...,...,...,...
5004,"[يحدث, تسخين]",العلوم,"[تسخين, تتسارع, حركة, جزيئاته, يؤدي, تمدده, وز..."
5005,"[الهدف, اتفاقية, التجارة, الحرة, لأمريكا]",الاقتصاد والعمل,"[الهدف, اتفاقية, التجارة, الحرة, لأمريكا, الشم..."
5006,"[الغرض, اتفاقية, التجارة, الحرة, لأمريكا]",الاقتصاد والعمل,"[الغرض, اتفاقية, التجارة, الحرة, لأمريكا, الشم..."
5007,"[هدف, إعجاز, القرآن, وفقا, للمعتقد]",الدين,"[يؤدي, الإعجاز, غرضين, رئيسين, الأول, أثبات, أ..."


In [24]:
df.to_csv("cleaned_data.csv", index=False, encoding="utf-8-sig")